# FlashSpec Colab 运行脚本

运行前先在 Colab 菜单选择 `Runtime -> Change runtime type -> GPU`，如果要验证 A100，硬件加速器选择 A100。下面的单元会安装项目、检查 CUDA/Triton、运行单测，并对比 dense、portable fused 和 Triton fused 三条 Kernel 1 路径。

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("当前 Colab runtime 没有 CUDA GPU，请先切换到 GPU/A100 runtime。")

## 1. 拉取代码并安装依赖

如果 `/content/FlashSpec` 已存在，就执行 `git pull`；否则从 GitHub clone。`.[triton]` 会安装 Triton 可选依赖，用于运行真实 Kernel 1。

In [ ]:
![ -d /content/FlashSpec/.git ] || git clone https://github.com/honey-floria/FlashSpec.git /content/FlashSpec
%cd /content/FlashSpec
!git pull
%pip install -e ".[triton]"

## 2. 检查 FlashSpec、CUDA 和 Triton 状态

`HAS_TRITON=True` 且 `cuda available=True` 时，`triton_fused` benchmark 会走真实 Triton fused dequant attention kernel。

In [ ]:
import sys
sys.path.insert(0, "/content/FlashSpec/src")

import torch
from flashspec.triton_kernels import HAS_TRITON

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("HAS_TRITON:", HAS_TRITON)

## 3. 运行正确性测试

这里会包含 CUDA + Triton 可用时的 Kernel 1 correctness 测试：输出对齐 PyTorch reference，并检查 `materializes_dense_kv == 0.0`。

In [ ]:
%cd /content/FlashSpec
!python -m unittest discover -s tests

## 4. Kernel 1 microbenchmark 对比

- `dense`: dense KV reference attention。
- `fused`: portable PyTorch INT8 KV 路径，会先 materialize dense KV。
- `triton_fused`: 真实 Kernel 1，直接读取 INT8 K/V，在 Triton kernel 内完成反量化、QK、softmax 和 PV。

In [ ]:
%cd /content/FlashSpec

# 小规模 smoke benchmark，先确认三条路径都能跑通。
!python benchmarks/microbench.py --backend dense --batch 4 --heads 8 --seq-len 512 --head-dim 64 --block-size 16 --iters 10 --device cuda --dtype float16 --json
!python benchmarks/microbench.py --backend fused --batch 4 --heads 8 --seq-len 512 --head-dim 64 --block-size 16 --iters 10 --device cuda --dtype float16 --json
!python benchmarks/microbench.py --backend triton_fused --batch 4 --heads 8 --seq-len 512 --head-dim 64 --block-size 16 --iters 10 --device cuda --dtype float16 --json

## 5. A100 目标 shape benchmark

TODO P0.1 要覆盖 `head_dim=64/128` 和 `seq_len=512/2048/4096`。下面先跑 Kernel 1 Triton fused 主路径，并把 JSON 保存到 `results/`。如果显存或 Colab 时间不够，可以减少 `--batch` 或 `--iters`。

In [ ]:
%cd /content/FlashSpec
!mkdir -p results

!python benchmarks/microbench.py --backend triton_fused --batch 16 --heads 32 --seq-len 512 --head-dim 64 --block-size 16 --iters 50 --device cuda --dtype float16 --json --output results/triton_fused_s512_d64.json
!python benchmarks/microbench.py --backend triton_fused --batch 16 --heads 32 --seq-len 2048 --head-dim 64 --block-size 16 --iters 50 --device cuda --dtype float16 --json --output results/triton_fused_s2048_d64.json
!python benchmarks/microbench.py --backend triton_fused --batch 16 --heads 32 --seq-len 4096 --head-dim 64 --block-size 16 --iters 50 --device cuda --dtype float16 --json --output results/triton_fused_s4096_d64.json

!python benchmarks/microbench.py --backend triton_fused --batch 16 --heads 32 --seq-len 512 --head-dim 128 --block-size 16 --iters 50 --device cuda --dtype float16 --json --output results/triton_fused_s512_d128.json
!python benchmarks/microbench.py --backend triton_fused --batch 16 --heads 32 --seq-len 2048 --head-dim 128 --block-size 16 --iters 50 --device cuda --dtype float16 --json --output results/triton_fused_s2048_d128.json
!python benchmarks/microbench.py --backend triton_fused --batch 16 --heads 32 --seq-len 4096 --head-dim 128 --block-size 16 --iters 50 --device cuda --dtype float16 --json --output results/triton_fused_s4096_d128.json

## 6. dense / portable fused / Triton fused 保存对比

这组命令用于同一个 shape 下对比三条路径。重点看 `latency_ms`、`tokens_per_second`、`compression_ratio` 和 `materializes_dense_kv`。

In [ ]:
%cd /content/FlashSpec
!mkdir -p results

!python benchmarks/microbench.py --backend dense --batch 16 --heads 32 --seq-len 2048 --head-dim 128 --block-size 16 --iters 50 --device cuda --dtype float16 --json --output results/dense_s2048_d128.json
!python benchmarks/microbench.py --backend fused --batch 16 --heads 32 --seq-len 2048 --head-dim 128 --block-size 16 --iters 50 --device cuda --dtype float16 --json --output results/portable_fused_s2048_d128.json
!python benchmarks/microbench.py --backend triton_fused --batch 16 --heads 32 --seq-len 2048 --head-dim 128 --block-size 16 --iters 50 --device cuda --dtype float16 --json --output results/triton_fused_s2048_d128_compare.json

## 7. 汇总结果

读取 `results/*.json`，快速查看每个 benchmark 的关键字段。

In [ ]:
import json
from pathlib import Path

for path in sorted(Path("/content/FlashSpec/results").glob("*.json")):
    data = json.loads(path.read_text())
    print(
        path.name,
        "backend=", data.get("backend"),
        "latency_ms=", round(float(data.get("latency_ms", 0.0)), 4),
        "tok/s=", round(float(data.get("tokens_per_second", 0.0)), 2),
        "compression=", round(float(data.get("compression_ratio", 0.0)), 3),
        "materializes_dense_kv=", data.get("materializes_dense_kv"),
    )